In [9]:
# RNN을 이용한 텍스트 생성
# 문맥을 반영해 다음 단어를 예측하여 텍스트 생성 (다항 분류)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences, to_categorical
import numpy as np
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.models import Sequential

text = """경마장에 있는 말이 뛰고 있다
그의 말이 법이다
가는 말이 고와야 오는 발이 곱다"""

# tok = Tokenizer(char_level=True)  # 글자 단위
# tok = Tokenizer(char_level=False)  # 단어 단위. 기본 값
tok = Tokenizer()
tok.fit_on_texts([text])
encoded = tok.texts_to_sequences([text])[0]
print(encoded)
print(tok.word_index)

vocab_size = len(tok.word_index) + 1  # 실제 단어 집합 + 1

# 훈련 데이터 작성
sequences = list()
for line in text.split('\n'):   # 문장 토큰화
    enco = tok.texts_to_sequences([line])[0]
    print(enco)
    # 바로 다음 단어를 label로 사용하기 위해 리스트에 담기
    for i in range(1, len(enco)):
      sequ = enco[:i + 1]
      # print(sequ)
      sequences.append(sequ)

print('학습에 참여할 샘플 수 :', len(sequences))
print(sequences)
print(max(len(i) for i in sequences))

# 전체 각각의 벡터 길이를 통일
max_len = max(len(i) for i in sequences)
psequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(psequences)

# 각 벡터의 마지막 요소를 label로 사용하기 위해 분리
x = psequences[:, :-1]  # feature
y = psequences[:, -1]   # label
print(x)
print(y)

# label은 one-hot encoding
y = to_categorical(y, num_classes=vocab_size)
print(y[:2])

[2, 3, 1, 4, 5, 6, 1, 7, 8, 1, 9, 10, 11, 12]
{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '발이': 11, '곱다': 12}
[2, 3, 1, 4, 5]
[6, 1, 7]
[8, 1, 9, 10, 11, 12]
학습에 참여할 샘플 수 : 11
[[2, 3], [2, 3, 1], [2, 3, 1, 4], [2, 3, 1, 4, 5], [6, 1], [6, 1, 7], [8, 1], [8, 1, 9], [8, 1, 9, 10], [8, 1, 9, 10, 11], [8, 1, 9, 10, 11, 12]]
6
[[ 0  0  0  0  2  3]
 [ 0  0  0  2  3  1]
 [ 0  0  2  3  1  4]
 [ 0  2  3  1  4  5]
 [ 0  0  0  0  6  1]
 [ 0  0  0  6  1  7]
 [ 0  0  0  0  8  1]
 [ 0  0  0  8  1  9]
 [ 0  0  8  1  9 10]
 [ 0  8  1  9 10 11]
 [ 8  1  9 10 11 12]]
[[ 0  0  0  0  2]
 [ 0  0  0  2  3]
 [ 0  0  2  3  1]
 [ 0  2  3  1  4]
 [ 0  0  0  0  6]
 [ 0  0  0  6  1]
 [ 0  0  0  0  8]
 [ 0  0  0  8  1]
 [ 0  0  8  1  9]
 [ 0  8  1  9 10]
 [ 8  1  9 10 11]]
[ 3  1  4  5  1  7  1  9 10 11 12]
[[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [12]:
# model
model = Sequential()
model.add(Embedding(vocab_size, 32))
model.add(LSTM(32, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

model.fit(x, y, epochs=200, verbose=2)
print(model.evaluate(x, y))

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/200
1/1 - 3s - 3s/step - accuracy: 0.1818 - loss: 2.5623
Epoch 2/200
1/1 - 0s - 49ms/step - accuracy: 0.3636 - loss: 2.5574
Epoch 3/200
1/1 - 0s - 45ms/step - accuracy: 0.3636 - loss: 2.5529
Epoch 4/200
1/1 - 0s - 44ms/step - accuracy: 0.2727 - loss: 2.5480
Epoch 5/200
1/1 - 0s - 45ms/step - accuracy: 0.2727 - loss: 2.5431
Epoch 6/200
1/1 - 0s - 43ms/step - accuracy: 0.2727 - loss: 2.5380
Epoch 7/200
1/1 - 0s - 45ms/step - accuracy: 0.2727 - loss: 2.5325
Epoch 8/200
1/1 - 0s - 48ms/step - accuracy: 0.2727 - loss: 2.5268
Epoch 9/200
1/1 - 0s - 46ms/step - accuracy: 0.2727 - loss: 2.5206
Epoch 10/200
1/1 - 0s - 43ms/step - accuracy: 0.2727 - loss: 2.5142
Epoch 11/200
1/1 - 0s - 45ms/step - accuracy: 0.2727 - loss: 2.5073
Epoch 12/200
1/1 - 0s - 43ms/step - accuracy: 0.2727 - loss: 2.4999
Epoch 13/200
1/1 - 0s - 44ms/step - accuracy: 0.2727 - loss: 2.4920
Epoch 14/200
1/1 - 0s - 45ms/step - accuracy: 0.2727 - loss: 2.4834
Epoch 15/200
1/1 - 0s - 43ms/step - accuracy: 0.2727 -

In [15]:
# 문자열 생성 함수
def sequence_gen_text(model, t, current_word, n):
  init_word = current_word
  sentence = ""
  for _ in range(n):
    encoded = t.texts_to_sequences([current_word])[0]
    encoded = pad_sequences([encoded], maxlen=max_len - 1, padding='pre')
    result = model.predict(encoded, verbose=0)
    result = np.argmax(result, axis=1)
    for word, index in t.word_index.items():
      # print(word, index)
      if index == result:
        break
    current_word = current_word + ' ' + word
    sentence = sentence + ' ' + word
  sentence = init_word + sentence
  return sentence

print(sequence_gen_text(model, tok, '경마장', 5))

경마장 말이 법이다 고와야 오는 발이
